# Von der Auto-Quote-API zu einem MCP-Server

Im vorherigen Notebook haben wir einen MCP-Server von Hand geschrieben: zwei Tools vor einer kleinen API. Das echte Backend von Auto Quote hat 23 Routen. Müssen wir für jede davon ein Tool schreiben?

Nein. Eine FastAPI-App beschreibt alle ihre Routen in einem OpenAPI-Dokument, also in JSON mit einem JSON-Schema für jeden Input. [FastMCP](https://gofastmcp.com) kann dieses Dokument lesen und daraus die Tools erzeugen: `FastMCP.from_fastapi(app)`.

1. Die ganze App in einer Zeile umwandeln
2. Das Resultat anschauen: einfach heisst nicht gut
3. Starten und Claude verbinden
4. Kuratieren: behalten, was ein Agent braucht, und es verständlich machen
5. Mit einem Agenten benutzen, dann Claude erneut verbinden

Dieses Notebook verwendet die **echte App**: Beim Import öffnet sie die Dateien in `backend/data/`, Sie sehen also Ihren eigenen Katalog und Ihre eigene Konfiguration. Es braucht dieselbe `.env.local` wie die App. Falls der Katalog leer ist, fügen Sie zuerst in der App ein paar Produkte hinzu.

In [ ]:
# Setup - Kann ignoriert werden
import functools
import json
import os
import threading
import time
from pathlib import Path

import uvicorn
from dotenv import load_dotenv
from fastmcp import Client, FastMCP
from fastmcp.server.providers.openapi import MCPType, RouteMap
from openai import OpenAI

MODEL = "gpt-6-luna"  # Text und Tool Calling

# Vom Arbeitsverzeichnis aufwärts bis zum Repo-Root (dem Ordner mit .git)
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())


@functools.cache
def get_client() -> OpenAI:
    load_dotenv(REPO_ROOT / ".env.local")
    return OpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.getenv("LLM_BASE_URL") or None)


client = get_client()

RUNNING_SERVERS: dict[int, tuple[uvicorn.Server, threading.Thread]] = {}


def serve(app, port: int) -> str:
    """Startet eine ASGI-App in einem Hintergrund-Thread und gibt ihre Basis-URL zurück. Wird die Zelle erneut ausgeführt, startet der Server neu."""
    stop(port)
    server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning"))
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    while not server.started:
        time.sleep(0.05)
    RUNNING_SERVERS[port] = (server, thread)
    return f"http://127.0.0.1:{port}"


def stop(port: int) -> None:
    if port in RUNNING_SERVERS:
        server, thread = RUNNING_SERVERS.pop(port)
        server.should_exit = True
        thread.join()

## 1. Die App und die Umwandlung in einer Zeile

`auto_quote.main` baut dieselbe FastAPI-`app`, die `make dev` startet. Wir starten sie nicht als Server: FastMCP ruft sie direkt im selben Prozess auf, es sind also weder ein Port noch ein Netzwerk beteiligt.

In [ ]:
from auto_quote.main import app

routes = [(method.upper(), path, operation["operationId"]) for path, methods in app.openapi()["paths"].items() for method, operation in methods.items()]
print(len(routes), "Routen")
for method, path, _ in routes:
    print(f"{method:6} {path}")

In [ ]:
mcp_all = FastMCP.from_fastapi(app, name="auto-quote")


async def show_tools(mcp: FastMCP) -> None:
    async with Client(mcp) as mcp_client:  # Ein Client kann direkt mit einem Server-Objekt sprechen
        tools = await mcp_client.list_tools()
    print(len(tools), "Tools")
    for tool in tools:
        print(f"- {tool.name}: {tool.description}")


await show_tools(mcp_all)

## 2. Einfach heisst nicht gut

Die Umwandlung hat funktioniert, und ein Agent könnte das alles bereits benutzen. Aber schauen Sie, was wir bekommen haben:

- **Unverständliche Namen**: `get_products_api_v1_products_get` wird aus der Python-Funktion, der URL und der Methode zusammengesetzt. Ein Modell wählt Tools nach Name und Beschreibung.
- **Keine Beschreibung**: Die Routen haben keinen Docstring, der einzige Text ist also ein Titel wie `Get Products`. Das Modell muss raten, wofür ein Tool gedacht ist.
- **Zu viel**: Datei-Uploads, ein PDF-Download und die Route für Dokumentdateien nützen einem Agenten nichts. Jedes zusätzliche Tool erschwert die Wahl.
- **Zu mächtig**: Auch die Routen `PUT`, `POST` und `DELETE` werden zu Tools. Damit kann ein Agent selbständig die Extraktionskonfiguration überschreiben oder den Dokumentindex löschen.

Die Generierung ist also der einfache Teil. **Auszuwählen, was bereitgestellt wird**, ist die eigentliche Arbeit, und es ist eine Entscheidung darüber, was Sie einem Agenten zutrauen.

## 3. Starten und Claude verbinden

Bevor wir etwas kuratieren, schauen wir uns an, wie sich die rohe Umwandlung von der anderen Seite anfühlt. Bisher haben wir den Server aus dem Notebook heraus aufgerufen. Ein MCP-Client wie **Claude Code** ist ein eigenes Programm, deshalb muss der Server auf einem Port lauschen. Die Zelle unten startet ihn im Hintergrund, wie die Server im vorherigen Notebook, und gibt den Befehl aus, der Claude Code damit verbindet.

> **Achtung: Dieser rohe Server stellt alle 23 Routen bereit, auch die schreibenden und löschenden, und zwar auf Ihren echten Daten in `backend/data/`.** Claude Code fragt nach, bevor es ein Tool ausführt: Lesen Sie den Namen des Tools und lehnen Sie alles ab, was nicht nur liest (`get_…`). Im nächsten Abschnitt kuratieren wir, und dieselbe URL liefert dann die sichere Version.


In [ ]:
MCP_URL = serve(mcp_all.http_app(), port=8003) + "/mcp"
print("MCP-Server läuft auf", MCP_URL)
print()
print("In einem Terminal mit Claude Code verbinden:")
print(f"  claude mcp add --transport http auto-quote {MCP_URL}")


Führen Sie den ausgegebenen Befehl in einem Terminal aus (nicht in diesem Notebook). Starten Sie dann eine **neue** Claude-Code-Session oder führen Sie `/mcp` in der aktuellen aus, und prüfen Sie, ob `auto-quote` verbunden ist. Fragen Sie zum Beispiel:

- *Welche Tools bekommst du von auto-quote?* Sie sehen die unverständlichen Namen aus Abschnitt 1.
- *Welche Produkte haben wir im Katalog?* Claude muss raten, welches der Tools es benutzen soll.

Der Server läuft nur so lange wie der Kernel dieses Notebooks. Um Claude Code später wieder zu trennen: `claude mcp remove auto-quote`.


## 4. Kuratieren

`from_fastapi` bietet dafür drei Optionen:

- `route_maps`: Regeln, die pro Methode und URL-Muster entscheiden, ob eine Route zu einem Tool wird oder ausgeschlossen wird. Die erste passende Regel gewinnt.
- `mcp_names`: neue Namen, mit der `operationId` der Route als Key (die dritte Spalte, die oben ausgegeben wurde).
- `mcp_component_fn`: ein Hook, der für jedes erzeugte Tool läuft, hier um eine Beschreibung zu setzen.

Wir behalten vier Routen, die nur lesen: den Produktkatalog, die Extraktionsfelder, das JSON-Schema und den Prompt, die Auto Quote daraus erzeugt (das Schema aus dem ersten Notebook), und den Status des Dokumentindex.

In [ ]:
EXPOSED_ROUTES = {
    "get_products_api_v1_products_get": (
        "list_products",
        "List every product of the catalogue with its reference, description, unit, unit price in CHF and specifications.",
    ),
    "get_extraction_configuration_api_v1_extraction_configuration_get": (
        "get_extraction_fields",
        "List the fields that are extracted from a customer request: key, label, type, whether it is required and its allowed values.",
    ),
    "preview_extraction_configuration_api_v1_extraction_configuration_preview_get": (
        "get_extraction_schema",
        "Get the JSON Schema and the prompt that are generated from the extraction fields and sent to the language model.",
    ),
    "get_document_index_status_api_v1_documents_index_get": (
        "get_document_index_status",
        "Get the status of the document index that is used to find similar past requests and offers.",
    ),
}


def describe(route, component) -> None:
    component.description = EXPOSED_ROUTES[route.operation_id][1]


exposed_paths = [path for method, path, operation_id in routes if operation_id in EXPOSED_ROUTES]

mcp = FastMCP.from_fastapi(
    app,
    name="auto-quote",
    route_maps=[
        RouteMap(methods=["GET"], pattern="^(" + "|".join(exposed_paths) + ")$", mcp_type=MCPType.TOOL),
        RouteMap(mcp_type=MCPType.EXCLUDE),
    ],
    mcp_names={operation_id: tool_name for operation_id, (tool_name, _) in EXPOSED_ROUTES.items()},
    mcp_component_fn=describe,
)

await show_tools(mcp)

Vier Tools statt 23, mit Namen und Beschreibungen, mit denen ein Modell arbeiten kann. Die Input-Schemas kommen ohne zusätzlichen Aufwand aus den Routen: Keines dieser vier Tools braucht Argumente, aber eine Route mit einem Pfadparameter oder einem Request-Body würde diese in `input_schema` zeigen.

Bevor ein Agent es benutzt, rufen wir selbst ein Tool auf, wie es ein Client tun würde. Der Request durchläuft die echte Route, die echte `CurrentUser`-Prüfung und den echten Controller.

In [ ]:
async with Client(mcp) as mcp_client:
    result = await mcp_client.call_tool("list_products")

print(json.dumps(result.structured_content["result"], indent=2, ensure_ascii=False)[:1500])

## 5. Mit einem Agenten benutzen

Dieselbe Schleife wie im vorherigen Notebook: Die MCP-Tools gehen als Function Tools an das Modell, das Modell fordert Aufrufe an, und wir führen sie auf dem MCP-Server aus.

In [ ]:
def to_openai_tool(tool) -> dict:
    return {"type": "function", "name": tool.name, "description": tool.description, "parameters": tool.input_schema}


async def ask_agent(question: str, max_steps: int = 5) -> str:
    async with Client(mcp) as mcp_client:
        tools = [to_openai_tool(tool) for tool in await mcp_client.list_tools()]
        messages = [{"role": "user", "content": question}]

        for _ in range(max_steps):
            response = client.responses.create(model=MODEL, input=messages, tools=tools)
            messages += response.output
            tool_calls = [item for item in response.output if item.type == "function_call"]
            if not tool_calls:
                return response.output_text

            for call in tool_calls:
                arguments = json.loads(call.arguments)
                print(f"tool call: {call.name}({arguments})")
                result = await mcp_client.call_tool(call.name, arguments, raise_on_error=False)
                output = "\n".join(block.text for block in result.content)
                messages.append({"type": "function_call_output", "call_id": call.call_id, "output": output})

    raise RuntimeError(f"Keine Antwort nach {max_steps} Schritten")

In [ ]:
print(await ask_agent("Welche Felder extrahieren wir aus einer Kundenanfrage, und welche davon sind Pflichtfelder?"))

In [ ]:
print(await ask_agent("Welche unserer Produkte haben einen Stundenpreis, und welches ist das günstigste?"))

Der Agent antwortet mit den Live-Daten der App. Wir haben kein Tool von Hand geschrieben: Die Routen, ihre Inputs und das JSON-Schema waren bereits vorhanden.

### Claude mit dem kuratierten Server verbinden

Der kuratierte Server ersetzt den rohen auf demselben Port, die URL bleibt also gleich. Wenn Sie Claude Code in Abschnitt 3 verbunden haben, verbinden Sie es neu (`/mcp` in Claude Code, oder starten Sie eine neue Session): Es listet jetzt vier Tools mit verständlichen Namen und Beschreibungen. Wenn Sie Abschnitt 3 übersprungen haben, führen Sie den ausgegebenen Befehl einmal aus.


In [ ]:
MCP_URL = serve(mcp.http_app(), port=8003) + "/mcp"
print("Kuratierter MCP-Server läuft auf", MCP_URL)
print(f"claude mcp add --transport http auto-quote {MCP_URL}")


Was Sie bedenken sollten, bevor Sie die App ausserhalb eines Workshops so bereitstellen:

- **Authentifizierung.** Der MCP-Server ruft die Routen der App auf, die `CurrentUser`-Prüfung läuft also weiterhin. Aber der `SessionCookieProvider` des Prototyps akzeptiert jeden Aufrufer und stellt eine neue Session aus, auf `localhost` schützt er also nichts. Standardmässig leitet FastMCP keine Credential-Header vom MCP-Client an die API weiter. Ein gemeinsam genutztes Deployment braucht einen echten `AuthProvider` und eine bewusste Entscheidung, wie sich der Agent identifiziert.
- **Schreibzugriffe.** Eine `POST`- oder `PUT`-Route zu `EXPOSED_ROUTES` hinzuzufügen ist eine Änderung von einer Zeile, und genau deshalb verdient sie ein Review. Offerten gehen an Kunden; ein Agent, der sie erstellen oder ändern kann, ist ein anderes Risiko als einer, der den Katalog lesen kann.
- **Kosten.** Routen, die das Sprachmodell aufrufen (zum Beispiel das Erstellen einer Extraktion), kosten bei jedem Aufruf Geld.
- **Andere APIs.** Für eine API, die keine FastAPI-App in Ihrem Prozess ist (ein CRM, ein ERP), macht `FastMCP.from_openapi(spec, client)` dasselbe aus ihrem OpenAPI-Dokument.

## Fazit

- Eine FastAPI-App beschreibt ihre Routen und deren Inputs bereits als JSON-Schema, die Umwandlung in MCP ist deshalb eine Zeile: `FastMCP.from_fastapi(app)`.
- Der erzeugte Server ist vollständig, aber nicht gut: unverständliche Namen, keine Beschreibungen, und die schreibenden Routen sind dabei.
- Die Arbeit liegt im Kuratieren: die Routen filtern (`route_maps`), die Tools benennen (`mcp_names`) und beschreiben (`mcp_component_fn`). Die Beschreibungen sind das, was das Modell liest.
- Was Sie bereitstellen, darf ein Agent tun. Beginnen Sie nur lesend, und behalten Sie Authentifizierung und Kosten im Blick.

## Herunterfahren

Die letzte Zelle stoppt den MCP-Server und alle anderen in diesem Notebook gestarteten Server. Ein Neustart oder das Schliessen des Kernels bewirkt dasselbe. Claude Code erfährt davon nichts: Es zeigt `auto-quote` als getrennt an, bis Sie es in einem Terminal mit `claude mcp remove auto-quote` entfernen.


In [ ]:
# Stoppt den in diesem Notebook gestarteten MCP-Server
for port in list(RUNNING_SERVERS):
    stop(port)
    print("Server auf Port", port, "gestoppt")
